In [0]:
%pip install confluent-kafka
dbutils.library.restartPython()

In [0]:
import importlib, kafka_conn
importlib.reload(kafka_conn)
from kafka_conn import get_kafka_conf, TOPIC
conf = get_kafka_conf(dbutils)
print("Config set. Topic:", TOPIC)

In [0]:
import json
from confluent_kafka import Consumer
from collections import defaultdict

TOPIC = "post-engagement"

consumer_conf = {
    **conf,                              # bootstrap + SASL_SSL from kafka_conn
    "group.id": "engagement-reader_2",     # this consumer's GROUP name (explained below)
    "auto.offset.reset": "earliest",     # a brand-new group starts at offset 0 (read all history)
    "enable.auto.commit": True,          # commit our offsets automatically as we go
}

consumer = Consumer(consumer_conf)
consumer.subscribe([TOPIC])

counts = defaultdict(lambda: {"view": 0, "like": 0})   # per-post tally

print("Reading post-engagement (will stop after ~10s of silence)...\n")
empty_polls = 0
try:
    while empty_polls < 20:              # ~10s with no new data -> stop (bounded, so the cell ends)
        msg = consumer.poll(0.5)        # wait up to 0.5s for a message
        if msg is None:
            empty_polls += 1
            continue
        if msg.error():
            print("error:", msg.error())
            continue

        empty_polls = 0
        event = json.loads(msg.value().decode("utf-8"))
        key   = msg.key().decode("utf-8")
        counts[key][event["type"]] += 1
        print(f"  p{msg.partition()} off {msg.offset():<3} key={key:7} type={event['type']}")
finally:
    consumer.close()                    # leaves the group cleanly + commits final offsets

print("\n--- Engagement tally per post ---")
for post in sorted(counts):
    print(f"  {post}:  views={counts[post]['view']}  likes={counts[post]['like']}")